# Phase 7 — Text-to-SQL AI Layer

## What This Phase Builds
A natural language interface that allows anyone to query 500,000 
real Medicare billing records without writing a single line of SQL.
Built on Groq's Llama API with full rate limiting and cost controls.

## How It Works
1. User types a question in plain English
2. Groq Llama AI converts it to a SQL query
3. SQL runs against cms_medicare.db
4. Results returned as a clean table

## Why This Matters
Phase 6 required SQL expertise to uncover fraud patterns. This layer 
removes that barrier entirely. A hospital administrator can ask 
"which providers in NJ are billing above $10,000?" and get an answer 
in seconds — no technical background needed.

## What We Found During Testing
The AI successfully surfaced every major fraud finding from Phase 5 
and Phase 6 through plain English questions alone — including 
Ellenberger's spinal surgery billing and Phoenix Eye's procedure 
mismatch. Three independent methods (Python anomaly detection, 
SQL scope analysis, AI natural language query) all pointing to 
the same providers.

## Limitations Discovered and Fixed
The AI initially confused provider last names with first names and 
used exact matching for organization names. Both were fixed by 
improving schema column descriptions and using LIKE for partial 
name searches. This reinforced that Text-to-SQL accuracy depends 
directly on schema documentation quality.

## Cost and Rate Limit Controls
- Groq Llama 3.3 70B — free tier, no credit card required
- Max 500 tokens per response — keeps costs minimal
- RateLimitController — proactively stays below 25 requests/minute
- Automatic retry logic — handles unexpected rate limit errors
- Session counter — caps usage at 50 requests per session

In [1]:
import sqlite3
import os
from dotenv import load_dotenv
from groq import Groq
import pandas as pd

# Load API keys
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=groq_api_key)

# Connect to database
conn = sqlite3.connect('data/cms_medicare.db')

# Cost control
MAX_QUERIES = 50
query_count = 0

print("Setup complete")
print(f"Groq API key loaded: {'Yes' if groq_api_key else 'No - check .env file'}")

Setup complete
Groq API key loaded: Yes


In [31]:
# Define the database schema
# This tells Claude exactly what table and columns exist
# The better we describe this, the more accurate the SQL will be

SCHEMA = """
You are a SQL expert analyzing Medicare billing data.

DATABASE: SQLite
TABLE: medicare_billing

COLUMNS:
- provider_id: unique identifier for each provider (NPI number)
- provider_name: last name or organization name. 
  Use LIKE '%name%' for partial matches, 
  use = 'exact name' only when you know the full exact name
- first_name: FIRST name only — do NOT use this to search for full names or surnames
- credentials: medical credentials (M.D., D.O., NP, PA etc.)
- entity_type: I = individual provider, O = organization
- address: street address
- city: city
- state: two letter state abbreviation (NY, CA, TX etc.)
- zip_code: 5 digit zip code
- country: country code
- specialty: medical specialty (Cardiology, Internal Medicine etc.)
- medicare_participant: Y = accepts Medicare, N = does not
- procedure_desc: plain English description of the procedure billed
- place_of_service: F = facility, O = office
- total_patients: number of unique patients billed for this procedure
- total_services: total number of times procedure was performed
- avg_submitted_charge: average amount provider submitted to Medicare
- avg_medicare_allowed: average amount Medicare decided procedure is worth
- avg_medicare_payment: average amount Medicare actually paid
- avg_standardized_payment: payment adjusted for geographic cost differences

IMPORTANT RULES:
- Always use LIMIT to prevent large result sets — default LIMIT 10
- Use ROUND() for decimal columns to 2 decimal places
- Use ORDER BY to make results meaningful
- The table has 500,000 rows so always aggregate when possible
- Never use SELECT * — always specify columns
"""

print("Schema defined successfully")
print(f"Schema length: {len(SCHEMA)} characters")

Schema defined successfully
Schema length: 1623 characters


In [32]:
import time

class RateLimitController:
    def __init__(self, requests_per_minute=25, daily_limit=900):
        # Set slightly below Groq limits as safety buffer
        # Groq allows 30/min and 1000/day - we use 25/min and 900/day
        self.requests_per_minute = requests_per_minute
        self.daily_limit = daily_limit
        self.minute_requests = []
        self.daily_requests = 0
        self.session_start = time.time()
        
    def can_make_request(self):
        now = time.time()
        
        # Clean requests older than 1 minute
        self.minute_requests = [t for t in self.minute_requests 
                                  if now - t < 60]
        
        # Check daily limit
        if self.daily_requests >= self.daily_limit:
            print(f"Daily limit reached ({self.daily_limit} requests)")
            return False
        
        # Check per minute limit
        if len(self.minute_requests) >= self.requests_per_minute:
            wait_time = 60 - (now - self.minute_requests[0])
            print(f"Approaching rate limit. Waiting {wait_time:.1f} seconds...")
            time.sleep(wait_time)
            
        return True
    
    def log_request(self):
        self.minute_requests.append(time.time())
        self.daily_requests += 1
        
    def status(self):
        now = time.time()
        recent = [t for t in self.minute_requests if now - t < 60]
        print(f"Requests this minute: {len(recent)}/{self.requests_per_minute}")
        print(f"Requests today:       {self.daily_requests}/{self.daily_limit}")
        print(f"Session duration:     {(now - self.session_start)/60:.1f} minutes")

# Initialize controller
rate_controller = RateLimitController()
print("Rate limit controller ready")

Rate limit controller ready


In [33]:
def text_to_sql(question, verbose=True):
    global query_count
    
    # Session limit check
    if query_count >= MAX_QUERIES:
        print("Session limit reached. Restart kernel to reset.")
        return None, None
    
    # Rate limit check BEFORE making API call
    if not rate_controller.can_make_request():
        print("Rate limit controller blocked this request.")
        return None, None
    
    # Build prompt
    prompt = f"""
{SCHEMA}

Convert this question to a SQL query:
"{question}"

Rules:
- Return ONLY the SQL query, nothing else
- No explanations, no markdown, no backticks
- Just the raw SQL query
"""
    
    # Retry loop for unexpected rate limit errors
    while True:
        try:
            message = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                max_tokens=500,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            
            # Log successful request
            rate_controller.log_request()
            
            # Extract and clean SQL
            sql = message.choices[0].message.content.strip()
            sql = sql.rstrip(';')
            query_count += 1
            
            if verbose:
                print(f"Question: {question}")
                print(f"\nGenerated SQL:")
                print(sql)
                print(f"\nQueries used: {query_count}/{MAX_QUERIES}")
                rate_controller.status()
            
            # Run SQL against database
            try:
                results = pd.read_sql_query(sql, conn)
                return sql, results
            except Exception as e:
                print(f"SQL Error: {e}")
                return sql, None
                
        except RateLimitError as e:
            wait_time = int(e.response.headers.get("retry-after", 5))
            print(f"Rate limit hit. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
            
        except Exception as e:
            print(f"Unexpected error: {e}")
            return None, None

print("Text-to-SQL with full rate limit control ready")

Text-to-SQL with full rate limit control ready


In [24]:
rate_controller.status()

Requests this minute: 0/25
Requests today:       0/900
Session duration:     0.1 minutes


In [13]:
# Test 2 - Geographic question
sql, results = text_to_sql("Which states have the most Medicare providers?")
if results is not None:
    print(results.to_string(index=False))

Question: Which states have the most Medicare providers?

Generated SQL:
SELECT state, COUNT(provider_id) as num_providers FROM medicare_billing GROUP BY state ORDER BY num_providers DESC LIMIT 10

Queries used: 3/50
Requests this minute: 1/25
Requests today:       1/900
Session duration:     1.2 minutes
state  num_providers
   CA          43036
   FL          37084
   TX          35312
   NY          32552
   PA          23480
   IL          20274
   OH          17787
   NC          17632
   MI          15748
   MA          14506


In [15]:
# Test - Fraud related
sql, results = text_to_sql(
    "Show me nurse practitioners with average charge above 5000"
)
if results is not None:
    print(results.to_string(index=False))

Question: Show me nurse practitioners with average charge above 5000

Generated SQL:
SELECT provider_name, ROUND(avg_submitted_charge, 2) AS avg_submitted_charge 
FROM medicare_billing 
WHERE credentials = 'NP' AND avg_submitted_charge > 5000 
GROUP BY provider_name 
ORDER BY avg_submitted_charge DESC 
LIMIT 10

Queries used: 4/50
Requests this minute: 2/25
Requests today:       2/900
Session duration:     1.8 minutes
provider_name  avg_submitted_charge
  Ellenberger              28056.08
   Stuedemann              17652.00
    Steinberg              13581.37
         Lamb              10688.00
       Vawter               8662.00
        Clubb               6155.53


In [17]:
# Test - Comparison question
sql, results = text_to_sql(
    "What is the difference between average submitted charge and Medicare payment for each specialty?"
)
if results is not None:
    print(results.to_string(index=False))

Question: What is the difference between average submitted charge and Medicare payment for each specialty?

Generated SQL:
SELECT specialty, ROUND(avg_submitted_charge, 2) AS avg_submitted, ROUND(avg_medicare_payment, 2) AS avg_payment, ROUND(avg_submitted_charge - avg_medicare_payment, 2) AS payment_diff FROM medicare_billing GROUP BY specialty ORDER BY payment_diff DESC LIMIT 10

Queries used: 5/50
Requests this minute: 3/25
Requests today:       3/900
Session duration:     2.1 minutes
                                     specialty  avg_submitted  avg_payment  payment_diff
                              Thoracic Surgery       13427.36       173.21      13254.15
                               Cardiac Surgery        6197.00      1129.15       5067.85
                    Ambulatory Surgical Center        4237.00        26.15       4210.85
                             Surgical Oncology        2757.00       646.57       2110.43
                      Undefined Physician type        2731.00 

In [26]:
# Test - Your own question from Phase 6 findings
sql, results = text_to_sql(
    "Show me all procedures billed by Ellenberger"
)
if results is not None:
    print(results.to_string(index=False))

Question: Show me all procedures billed by provider with last name Ellenberger

Generated SQL:
SELECT procedure_desc, ROUND(avg_submitted_charge, 2), ROUND(avg_medicare_allowed, 2), ROUND(avg_medicare_payment, 2) 
FROM medicare_billing 
WHERE provider_name = 'Ellenberger' 
ORDER BY avg_medicare_payment DESC 
LIMIT 10

Queries used: 9/50
Requests this minute: 2/25
Requests today:       2/900
Session duration:     0.4 minutes
                                                                   procedure_desc  ROUND(avg_submitted_charge, 2)  ROUND(avg_medicare_allowed, 2)  ROUND(avg_medicare_payment, 2)
        Fusion of spine in lower back with partial removal of spine bone and disc                        28056.08                          262.21                          208.92
                      New patient office or other outpatient visit, 30-44 minutes                          350.00                          101.72                           79.92
              Established patient offi

In [27]:
# Test 1 - Geographic fraud pattern we found in Phase 6
sql, results = text_to_sql(
    "Show me providers in New Jersey with average charge above 10000"
)
if results is not None:
    print(results.to_string(index=False))

Question: Show me providers in New Jersey with average charge above 10000

Generated SQL:
SELECT provider_name, ROUND(avg_submitted_charge, 2) AS avg_charge 
FROM medicare_billing 
WHERE state = 'NJ' AND avg_submitted_charge > 10000 
GROUP BY provider_name 
ORDER BY avg_submitted_charge DESC 
LIMIT 10

Queries used: 10/50
Requests this minute: 1/25
Requests today:       3/900
Session duration:     2.8 minutes
               provider_name  avg_charge
                   Buonocore    79714.29
Freedom Surgical Center, Llc    52820.84
                   Drillings    52500.00
                       Patel    40000.00
                      Sheikh    34800.00
                  Degregorio    34113.00
                     Richard    32830.91
                        Yrad    32352.94
                     Gribbin    31181.92
                   Radvinsky    29815.90


In [34]:
# Test 2 - Phoenix Eye finding from Phase 6
sql, results = text_to_sql(
    "Show me all procedures billed by Phoenix Eye Surgical Center"
)
if results is not None:
    print(results.to_string(index=False))

Question: Show me all procedures billed by Phoenix Eye Surgical Center

Generated SQL:
SELECT procedure_desc, ROUND(avg_medicare_payment, 2) AS avg_medicare_payment, total_services 
FROM medicare_billing 
WHERE provider_name LIKE '%Phoenix Eye Surgical Center%' 
AND entity_type = 'O' 
GROUP BY procedure_desc 
ORDER BY total_services DESC 
LIMIT 10

Queries used: 14/50
Requests this minute: 1/25
Requests today:       1/900
Session duration:     0.1 minutes
                                                  procedure_desc  avg_medicare_payment  total_services
       Insertion of sacral nerve neurostimulator electrode array               3863.17            76.0
Insertion of spinal neurostimulator electrode array through skin               3677.67            60.0
       Insertion of spinal neurostimulator generator or receiver              18669.18            35.0
    Insertion of peripheral or gastric neurostimulator generator              14973.11            12.0


In [29]:
# Test 3 - Benchmark question
sql, results = text_to_sql(
    "What is the average Medicare payment for office visits?"
)
if results is not None:
    print(results.to_string(index=False))

Question: What is the average Medicare payment for office visits?

Generated SQL:
SELECT ROUND(AVG(avg_medicare_payment), 2) AS avg_medicare_payment 
FROM medicare_billing 
WHERE place_of_service = 'O' 
LIMIT 10

Queries used: 12/50
Requests this minute: 3/25
Requests today:       5/900
Session duration:     3.3 minutes
 avg_medicare_payment
                 68.4


In [30]:
# Test 4 - Complex question
sql, results = text_to_sql(
    "Which specialties in California have more than 100 providers and high average charges?"
)
if results is not None:
    print(results.to_string(index=False))

Question: Which specialties in California have more than 100 providers and high average charges?

Generated SQL:
SELECT specialty, COUNT(provider_id) AS num_providers, ROUND(AVG(avg_submitted_charge), 2) AS avg_charge 
FROM medicare_billing 
WHERE state = 'CA' 
GROUP BY specialty 
HAVING COUNT(provider_id) > 100 
ORDER BY avg_charge DESC 
LIMIT 10

Queries used: 13/50
Requests this minute: 4/25
Requests today:       6/900
Session duration:     3.6 minutes
                                     specialty  num_providers  avg_charge
                    Ambulatory Surgical Center            243     8317.31
                      Interventional Radiology            317     1827.93
                                  Neurosurgery            200     1642.72
                              Vascular Surgery            261     1270.97
                                Anesthesiology           1002     1230.44
 Certified Registered Nurse Anesthetist (CRNA)            187     1134.43
                      

## Phase 7 — Text-to-SQL AI Layer

In this phase we added an AI layer that allows both technical and 
non-technical professionals to query the Medicare billing database 
using plain English. A hospital administrator, fraud investigator, 
or policy maker can now ask questions without knowing any SQL.

The Text-to-SQL layer takes a natural language question, converts it 
to a SQL query using the Groq Llama API, runs it against the 
cms_medicare.db database, and returns results as a clean table. 
The AI is given a detailed schema describing every column so it 
understands the data structure before generating queries.

This layer connects directly to the fraud findings from Phase 5 
and Phase 6. Asking "show me nurse practitioners billing above 5000" 
immediately surfaces Ellenberger — the same provider flagged by 
z-score anomaly detection in Python and scope violation detection 
in SQL. Three independent methods pointing to the same provider.

During testing I identified two limitations. First, the AI confused 
provider_name (last name) with first_name when searching by surname. 
Second, it used exact matching when partial matching was needed for 
organization names. Both were fixed by improving the schema 
descriptions and using LIKE for partial name searches. This taught 
me that Text-to-SQL accuracy depends directly on how well the 
database schema is documented.

Rate limiting is handled through a three layer protection system — 
a proactive RateLimitController that stays below Groq limits, 
automatic retry logic for unexpected rate limit errors, and a 
session query counter capping usage at 50 requests.